# 第14周 · 作业：正式研究实验 3（预计约7小时）

**研究问题**：在 IMDB 情感分类上，四种"使用预训练模型"的方式——**只训分类头**（线性探针式）、**全量微调**、**LoRA r=4**、**LoRA r=16**——在测试准确率、可训练参数量与训练时间上如何取舍？它们能否赢过 TF-IDF 基线？

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 实验设计：超参数分类 + 预先写下的假设 | 0.5 小时 |
| B | 执行：4 种方法 × 3 个种子（Colab T4）+ TF-IDF 基线 | 2.5 小时（大部分是等 GPU） |
| C | 结果表、参数拆账、图 + 分析问题 | 1.5 小时 |
| D | 实验记录（正式格式） | 1 小时 |
| E | 代码讲解 | 0.5 小时 |
| — | 继续你的论文复现（下周写复现报告，模板 `templates/replication_report.md`） | 1 小时 |

**在 Colab T4 上做这份作业**（代码执行程序 → 更改运行时类型 → T4 GPU）。下面的配置格会自动识别 GPU，切换到正式配置：训练 5000 条、最长 256 个 token、2 个 epoch、混合精度、3 个种子。在 CPU 上运行时自动换成**缩小配置**（训练 300 条、最长 96 个 token、1 个种子）——它只用来检查代码能跑通，**分析与实验记录必须用 T4 上的结果写**。

**约定**：可以使用 AI 编程助手，但一定要写 E 部分的"代码讲解"。**自评**（学习指南里的研究实验自评表）：设计 30% · 执行 30% · 分析与解释 30% · 表达 10%。

选做题（小型 ViT 微调 CIFAR-10 子集）在本 notebook 最后。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 第 14 周追加：Hugging Face transformers（5.x 写法）与 PEFT。Colab 预装了 transformers，一般只需装 peft
for pkg, spec in (("transformers", "transformers>=5.22,<6"), ("peft", "peft>=0.21,<1")):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
print("transformers / peft 已就绪")

In [ ]:
import os, time, math
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from zero2researcher import data, experiments
from zero2researcher.checks import week14 as ck

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # 本周跳过 MPS（见讲义）
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

## A · 实验设计（先写，再跑）

实验规格：

- **科学变量**：适配方式 ∈ {只训分类头 `probe`，全量微调 `full`，LoRA r=4 `lora4`，LoRA r=16 `lora16`}；
- **数据**：课程 IMDB 子集，训练集前 `N_TRAIN` 条，测试集前 `N_TEST` 条（T4：5000 / 2000）；
- **固定**：`distilbert-base-uncased`、最长 `MAX_LEN` 个 token、batch 16、AdamW、`EPOCHS` 个 epoch；LoRA 作用于 `q_lin`、`v_lin`，`lora_alpha = 8`（两种秩相同），`lora_dropout = 0.1`。α 的取法照 LoRA 论文 §4.1：α 是与 r 无关的常数，取定后不再调（论文把它设为第一次尝试的 r）；我们第一次尝试的是讲义 S2 的 r = 4、α = 8，所以 r = 16 也用 α = 8，缩放 α/r 分别为 2 和 0.5。论文说这种缩放能减少换 r 时重调超参数（如学习率）的需要——这正是它这样约定的理由。（社区里常见的 `lora_alpha = 2r` 让 α/r 恒为 2，等于换 r 时同时换了等效步长；它不是论文的约定。）；
- **学习率**（每种方法一个，取自文献的常用值，**本实验不调**）：全量微调 2e-5（HF 教程）；LoRA 5e-4（LoRA 论文附录 D 表 9 对 RoBERTa-base 用 4e-4–5e-4）；只训分类头 1e-3；
- **种子** 0、1、2：报告测试准确率的均值 ± 标准差；同时记录**训练用时**和**可训练参数量**（并拆成"分类头"与"LoRA A/B"两部分）；
- **基线**：TF-IDF + 逻辑回归，在同一份训练集与测试集上；外加第 6 周口径（5000 条训练）的那一行。

"只训分类头"的实现方式：DistilBERT 身体完全冻结，所以把每条影评 `[CLS]` 位置的最后一层表示**一次算好、缓存起来**，再只训练分类头（`pre_classifier` + `classifier`）。这与"冻结身体、每个 epoch 都重新前向"在数学上等价（只差身体里的 dropout），但快得多——它的训练用时要把"算特征"的时间也算进去。

**A1 · 超参数分类**（科学 / 冗余 / 固定，各至少写两个；本实验对冗余超参数是怎么处理的？这样做有什么风险？）

- 科学超参数：
- 冗余超参数：
- 固定超参数：

**A2 · 假设**（四种方法的准确率排序你预期如何？LoRA 的可训练参数比全量微调少多少倍，你预期准确率差多少？谁能赢 TF-IDF？什么样的结果会推翻你的假设？）

-

## B · 执行实验

### B1 · 配置与数据（已给出）

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"   # 固定模型版本（Hub 上的提交号）：模型仓库以后更新，课程里的数字也能复现
ON_GPU = torch.cuda.is_available()
if ON_GPU:      # Colab T4：作业要求的正式配置
    N_TRAIN, N_TEST, MAX_LEN, EPOCHS, SEEDS = 5000, 2000, 256, 2, (0, 1, 2)
else:           # CPU：缩小配置，只检查代码能跑通（结论用 T4 的结果写）
    N_TRAIN, N_TEST, MAX_LEN, EPOCHS, SEEDS = 300, 500, 96, 3, (0,)
USE_AMP = ON_GPU                    # T4 上开混合精度（fp16）：更快、更省显存
BATCH = 16
METHODS = ["probe", "full", "lora4", "lora16"]
LR = {"probe": 1e-3, "full": 2e-5, "lora4": 5e-4, "lora16": 5e-4}
print(f"配置：train {N_TRAIN} | test {N_TEST} | max_len {MAX_LEN} | epochs {EPOCHS} | seeds {SEEDS} | fp16 {USE_AMP}")

tok = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
imdb_train, imdb_test = data.load_imdb("train"), data.load_imdb("test")
train_df, test_df = imdb_train.iloc[:N_TRAIN], imdb_test.iloc[:N_TEST]


def encode(texts):
    enc = tok(list(texts), truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]


ids_tr, mask_tr = encode(train_df["text"])
ids_te, mask_te = encode(test_df["text"])
y_tr = torch.tensor(train_df["label"].to_numpy())
y_te = torch.tensor(test_df["label"].to_numpy())
print("训练", tuple(ids_tr.shape), "| 测试", tuple(ids_te.shape),
      f"| 正例比例 {y_tr.float().mean().item():.3f} / {y_te.float().mean().item():.3f}")


def autocast():
    # T4 上用 fp16 自动混合精度；CPU 上是一个什么也不做的上下文
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP)


def evaluate(model, ids, mask, y, batch_size=64):
    model.eval()
    correct = 0
    with torch.no_grad(), autocast():
        for i in range(0, len(y), batch_size):
            logits = model(input_ids=ids[i:i + batch_size].to(device),
                           attention_mask=mask[i:i + batch_size].to(device)).logits
            correct += (logits.argmax(dim=1).cpu() == y[i:i + batch_size]).sum().item()
    return correct / len(y)


def cls_features(model, ids, mask, batch_size=64):
    # 冻结的 DistilBERT 身体 → 每条影评 [CLS] 位置的最后一层表示 (N, 768)，分类头读的就是它
    model.eval()
    feats = []
    with torch.no_grad(), autocast():
        for i in range(0, len(ids), batch_size):
            h = model.distilbert(input_ids=ids[i:i + batch_size].to(device),
                                 attention_mask=mask[i:i + batch_size].to(device)).last_hidden_state
            feats.append(h[:, 0].float())
    return torch.cat(feats)


def head_logits(model, feats):
    # DistilBertForSequenceClassification 分类头的前向：pre_classifier → ReLU → dropout → classifier
    return model.classifier(model.dropout(F.relu(model.pre_classifier(feats))))

### B2 · 建模型：四种方法各自"哪些参数可训练"

写 `build_model(method)`：加载一个新的 `AutoModelForSequenceClassification`（`num_labels=2`），然后

- `"probe"`：冻结全部参数，只打开 `pre_classifier` 与 `classifier`；
- `"full"`：什么都不冻结；
- `"lora4"` / `"lora16"`：用 PEFT 的 `LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=8, lora_dropout=0.1, target_modules=["q_lin", "v_lin"])` 包装（两种秩都用 α = 8，见 A 部分）。

再写 `param_breakdown(model)`：返回 `(trainable, head, lora)` 三个整数——可训练参数总数、其中属于分类头的、其中属于 LoRA A/B 矩阵的（参数名里含 `"lora_"`）。全量微调时 `trainable` 远大于 `head + lora`，多出来的是身体。

In [ ]:
def build_model(method):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=REVISION, num_labels=2)
    # 在这里写你的代码
    raise NotImplementedError
    return model.to(device)


def param_breakdown(model):
    # 在这里写你的代码
    raise NotImplementedError


breakdown = {}
for method in METHODS:
    m = build_model(method)
    breakdown[method] = trainable, head, lora = param_breakdown(m)
    print(f"{method:7s} 可训练 {trainable:>11,} = 分类头 {head:,} + LoRA A/B {lora:,} + 身体 {trainable - head - lora:,}")
    if method == "full":
        total_params = sum(p.numel() for p in m.parameters())
    if method.startswith("lora"):
        m.print_trainable_parameters()
        ck.peft_accounting(trainable, head, lora, m)     # 传入模型：自查会独立重数一遍再对账
print(f"全模型参数 {total_params:,}")

**B2 追问**（写进 C 部分的参数表）：`print_trainable_parameters` 报的 r=4 的数里，LoRA 本身占多少？用公式 6 层 × 2 个模块 × $r(768 + 768)$ 验算 `lora` 那一列。

### B3 · 训练一种方法、一个种子

写 `run_method(method, seed)`：固定种子 → `build_model` → 训练 `EPOCHS` 个 epoch → 返回 `{"acc": 测试准确率, "seconds": 训练用时}`。

- `"probe"`：用 `cls_features` 算出训练集的特征（**计入训练用时**），再用 `head_logits` 在缓存特征上训练分类头（batch 16、AdamW、`LR["probe"]`、交叉熵）；计时结束后再算测试集特征、求准确率；
- 其余三种：第 8 周的五步循环（`model(input_ids=..., attention_mask=..., labels=...)` 的 `.loss`），优化器只拿 `requires_grad` 的参数；
- 混合精度的骨架已经写在下面的格里（已经给出）：前向包在 `with autocast():` 里，`loss.backward()` 换成 `scaler.scale(loss).backward()`，`optimizer.step()` 换成 `scaler.step(optimizer); scaler.update()`——你只需要在 `autocast` 里写出前向与损失。CPU 上 `scaler` 是关闭的，这几行等价于普通写法。

In [ ]:
def run_method(method, seed):
    experiments.set_seed(seed)
    model = build_model(method)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR[method])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    t0 = time.time()
    if method == "probe":
        # 只训分类头：先用 cls_features 算训练集特征（计入用时），再用 head_logits 在缓存特征上训练分类头；
        # 计时结束后再算测试集特征，求出 acc 与 seconds
        # 在这里写你的代码
        raise NotImplementedError
        return {"acc": acc, "seconds": seconds}

    # 其余三种：第 8 周的五步循环 + 混合精度（骨架已给出）
    loader = DataLoader(TensorDataset(ids_tr, mask_tr, y_tr), batch_size=BATCH, shuffle=True)
    for _ in range(EPOCHS):
        model.train()
        for ids, mask, yb in loader:
            with autocast():                       # 前向与损失在 autocast 里算（T4 上是 fp16）
                # 在这里写你的代码
                raise NotImplementedError
            optimizer.zero_grad()
            scaler.scale(loss).backward()          # 代替 loss.backward()：先把损失放大，防止 fp16 梯度下溢
            scaler.step(optimizer)                 # 代替 optimizer.step()：先把梯度缩回原尺度再更新
            scaler.update()
    seconds = time.time() - t0
    return {"acc": evaluate(model, ids_te, mask_te, y_te), "seconds": seconds}

### B4 · 主循环

对每种方法、每个种子调用 `run_method`，结果存成下面的结构（自查会检查）：

```python
results[method] = {"accs": [每个种子的测试准确率], "seconds": [每个种子的训练用时],
                   "trainable": 可训练参数, "head": 其中分类头, "lora": 其中 LoRA A/B}
```

T4 上 4 种方法 × 3 个种子预计要二三十分钟；CPU 缩小配置约两三分钟。

In [ ]:
results = {}
t_all = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"全部 {len(METHODS) * len(SEEDS)} 次训练用时 {time.time() - t_all:.0f} 秒")
for method, r in results.items():
    print(f"{method:7s} 测试准确率 {experiments.format_result(r['accs'])} | 训练用时 {np.mean(r['seconds']):.1f} 秒")
ck.adaptation_experiment(results, n_seeds=len(SEEDS))
ck.finetune_accuracy(np.mean(results["full"]["accs"]), config="t4" if ON_GPU else "cpu")

### B5 · TF-IDF 基线

两行：①同一份训练集（`N_TRAIN` 条）、同一份测试集；②第 6 周口径——用全部 5000 条训练，测同一份测试集。

In [ ]:
def tfidf_accuracy(train_texts, train_labels, test_texts, test_labels):
    vec = TfidfVectorizer(min_df=2)
    clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(train_texts), train_labels)
    return clf.score(vec.transform(test_texts), test_labels)


# 在这里写你的代码
raise NotImplementedError
print(f"TF-IDF + LR：训练 {N_TRAIN} 条 → {tfidf_same:.4f} | 训练 5000 条 → {tfidf_5000:.4f}（同一份测试集，{N_TEST} 条）")

## C · 结果表、参数拆账与图

**C1** · 结果表 `table`（pandas DataFrame）：每种方法一行（外加两行 TF-IDF），列为：测试准确率（均值 ± 标准差）、可训练参数、其中分类头、其中 LoRA A/B、可训练参数占全模型的百分比、训练用时（秒，均值）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
table

**C2** · 一张两栏的图：左栏横轴是可训练参数（对数轴）、纵轴是测试准确率（误差棒 = 标准差），四种方法各一个点并标注名字，两条 TF-IDF 画成水平虚线；右栏是四种方法的训练用时柱状图。标题写结论。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**C3 · 分析问题**（每题两三句话，引用你 **T4 结果**表里的数字）

1. LoRA（r=4、r=16）与全量微调的准确率之差，是否大于种子间的标准差？用第 6 周的三分法下结论。可训练参数相差多少倍？
2. r=4 与 r=16 有区别吗？LoRA 的 A/B 参数涨了 4 倍，总可训练参数涨了多少倍？为什么（参数拆账）？注意两种秩都用 α = 8，r=16 的缩放 α/r 只有 r=4 的 1/4——这是 LoRA 论文的约定，你比较的是"按论文约定换 r"的效果；若要单独看秩的作用，还需要什么对照？
3. 只训分类头为什么最便宜？它的"训练用时"主要花在哪一步？它的准确率与上周 mean-pool 探针（0.8625）相比如何，你能想到什么原因？
4. 哪些方法赢过了"同样数据量"的 TF-IDF？哪些赢过了 5000 条训练的 TF-IDF？如果只看 CPU 缩小配置的结果，你会得出什么**错误**结论？
5. 本实验没有为每种方法调学习率（冗余超参数用了文献默认值）。这会怎样影响结论的可信度？如果只能多跑 6 次训练，你会怎样设计一个最小的补救实验？
6. 训练时间的比较公平吗？（提示：probe 的特征缓存、LoRA 的反向传播仍要穿过整个网络、fp16 对不同方法的加速是否相同。）

（在这里回答）

## D · 实验记录（正式格式）

"结果"一栏必须是 T4 上 3 个种子的均值 ± 标准差，并引用 C 部分的表和图；"局限"一栏至少讨论学习率未调与固定超参数（epochs、max_len）对结论的限定。

## 实验记录 #3 · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## E · 代码讲解（每份作业都写）

用你自己的话写一段（约 150–300 字）：

1. 逐段解释 `build_model`、`param_breakdown` 与 `run_method`——尤其是：**哪些参数在每种方法里可训练、你是怎么确认的**；混合精度那几行（`autocast`、`GradScaler`）各在做什么；
2. 注明哪些代码由 AI 助手生成、你对它做了哪些检查或修改（例如：PEFT 是否把分类头设为可训练、优化器是否只拿到了可训练参数、`model.eval()` 是否在评估前调用）；
3. 写出一处你一开始没看懂、后来弄懂的地方。

（在这里写）

## 选做 · 在 CIFAR-10 子集上微调一个小型 ViT

（选做；需要 Colab T4。下面的代码**只作为参考写法**，不在本 notebook 里运行。）

ViT（Dosovitskiy et al. 2020, arXiv:2010.11929）把图片切成 16×16 的小块、当成"词"送进 Transformer——和 DistilBERT 是同一种编码器。HF 官方教程 [Image classification](https://huggingface.co/docs/transformers/tasks/image_classification) 用的是 `google/vit-base-patch16-224-in21k`（8600 万参数）；更小的社区版本 `WinKawaks/vit-tiny-patch16-224`（约 570 万参数，Apache-2.0）在 T4 上更快。要点：

1. CIFAR-10 的图是 32×32，ViT 要 224×224：用模型自带的 `AutoImageProcessor` 缩放和归一化（这一步比训练本身还费时，建议子集 ≤ 5000 张，1–2 个 epoch）；
2. `AutoModelForImageClassification.from_pretrained(..., num_labels=10, ignore_mismatched_sizes=True)`——原模型的头是 1000 类，换成 10 类；
3. 训练循环与本作业的 `run_method` 完全相同（输入换成 `pixel_values`）；也可以对 ViT 的 `query`、`value` 层加 LoRA，比较"全量 vs LoRA"在图像上是否同样成立。

参考写法（复制到一个新的 Colab notebook 里运行）：

```python
from torchvision import datasets
from transformers import AutoImageProcessor, AutoModelForImageClassification

name = "WinKawaks/vit-tiny-patch16-224"
processor = AutoImageProcessor.from_pretrained(name)
cifar = datasets.CIFAR10(root=data.download_root(), train=True, download=True)
images = [cifar[i][0] for i in range(5000)]                      # PIL 图片
labels = torch.tensor([cifar[i][1] for i in range(5000)])
pixel_values = processor(images=images, return_tensors="pt")["pixel_values"]   # (5000, 3, 224, 224)

model = AutoModelForImageClassification.from_pretrained(name, num_labels=10, ignore_mismatched_sizes=True).to("cuda")
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
loader = DataLoader(TensorDataset(pixel_values, labels), batch_size=32, shuffle=True)
for epoch in range(2):
    model.train()
    for xb, yb in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(pixel_values=xb.to("cuda"), labels=yb.to("cuda")).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
# 测试集同样处理后评估；与第 10 周从零训练的 CNN 比较
```

产出：一张表（ViT 微调 vs 第 10 周 CNN 的测试准确率，均值 ± 标准差）与三句话的结论。